In [4]:
#!pip install youtube-transcript-api langchain-community langchain-openai faiss-cpu tiktoken python-dotenv

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda

In [12]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [26]:
video_id = "Gfr50f6ZBvo"   # ONLY the id, not the full URL

try:
    ytt_api = YouTubeTranscriptApi()
    fetched = ytt_api.fetch(video_id, languages=["en"])
    transcript = " ".join(snippet.text for snippet in fetched)
    print(transcript[:500]) 
except TranscriptsDisabled:
    print("No captions available for this video.")

the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful 


In [28]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunks = splitter.create_documents([transcript])
len(chunks)

168

In [30]:
chunks[12]

Document(metadata={}, page_content="i mean all machines do that to some extent they all enhance our natural capabilities obviously cars make us allow us to move faster than we can run but this was a machine to extend the mind and and then of course ai is the ultimate expression of what a machine may be able to do or learn so very naturally for me that thought extended into into ai quite quickly remember the the programming language that was first started special to the machine no it was just the base it was just i think it was just basic uh on the zx spectrum i don't know what specific form it was and then later on i got a commodore amiga which uh was a fantastic machine no you're just showing off so yeah well lots of my friends had atari st's and i i managed to get amigas it was a bit more powerful and uh and that was incredible and used to do um programming in assembler and and uh also amos basic this this specific form of basic it was incredible actually as well all my coding skills

In [33]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vector_store = FAISS.from_documents(chunks, embeddings)

Loading weights: 100%|████████| 103/103 [00:00<00:00, 15129.70it/s]


In [34]:
vector_store.index_to_docstore_id

{0: 'b86f3c33-66f6-4e17-93b3-2864d2932820',
 1: '51c9ac5e-7305-48dc-999f-f2dfec0b3ad7',
 2: '1ab0922b-3221-4faa-9515-6522bf6f23bc',
 3: '107389a2-98c2-4cd5-8f04-2380b1cfa048',
 4: '91855886-258b-420c-922a-75a7b92d7e97',
 5: '2dc765dc-82e5-4494-91b5-ccb51855df8f',
 6: 'd50402b0-6d9a-424a-a972-3ef604845499',
 7: 'ba0353a8-48ad-44c8-8595-d3152917b13c',
 8: '7d93bf00-e62e-456f-b033-8935e81e0c9d',
 9: '1fa05758-34f1-453d-b66c-f89e21f988b9',
 10: 'b4b2f6c2-3ad8-4805-bd6e-9a6c4a4eda0b',
 11: '05426d2f-6d0c-4ba7-970a-c9fbfc56859d',
 12: '77cbe656-c832-4635-91e2-aaa8f8086c00',
 13: '1b1eec19-ce35-492a-8b46-1af0ec1c7af7',
 14: 'f31c159c-4903-4ffa-86f9-a5a6a4e92daf',
 15: '971e9410-0efa-489d-8c53-190932a8b0df',
 16: 'bc2eb5ff-3f2a-4e27-bc21-16dc0dea34bd',
 17: 'b97f8566-3c17-453b-a464-6cfbbc767cd5',
 18: '9e1599c5-c719-4804-ad07-92eb06a20f87',
 19: 'e8da8e3c-392c-4793-ae7a-6c27e171fb9f',
 20: '57550cd1-96ad-4169-ba1d-569d1a0e5c04',
 21: '1628f8b6-4529-4083-a211-98e80fa313f1',
 22: 'd604d29b-350b-

In [35]:
vector_store.get_by_ids(['a1168369-d004-49bb-8991-271de7c07c87'])

[Document(id='a1168369-d004-49bb-8991-271de7c07c87', metadata={}, page_content="far i think most neuroscientists and most mainstream biologists and neuroscientists would say there's no evidence of any quantum uh systems or effects in the brain as far as we can see it's it can be mostly explained by classical uh classical theories so and then so there's sort of the the search from the biology side and then at the same time there's the raising of the water uh at the bar from what classical turing machines can do uh uh and and you know including our new ai systems and uh as you alluded to earlier um you know i think ai especially in the last decade plus has been a continual story now of surprising uh events uh and surprising successes knocking over one theory after another of what was thought to be impossible you know from go to protein folding and so on and so i think um i would be very hesitant to bet against how far the uh universal turing machine and classical computation paradigm can

In [37]:
retriever = vector_store.as_retriever(search_type='similarity',search_kwargs={"k":4})

In [39]:
retriever.invoke("what is Deepmind")

[Document(id='904c9c78-6be9-4f8a-aa03-b19a0e85ef8f', metadata={}, page_content="and how it works this is tough to uh ask you this question because you probably will say it's everything but let's let's try let's try to think to this because you're in a very interesting position where deepmind is the place of some of the most uh brilliant ideas in the history of ai but it's also a place of brilliant engineering so how much of solving intelligence this big goal for deepmind how much of it is science how much is engineering so how much is the algorithms how much is the data how much is the hardware compute infrastructure how much is it the software computer infrastructure yeah um what else is there how much is the human infrastructure and like just the humans interact in certain kinds of ways in all the space of all those ideas how much does maybe like philosophy how much what's the key if um uh if if you were to sort of look back like if we go forward 200 years look back what was the key 

In [56]:
llm_endpoint = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="conversational",
    provider="novita",    
)
llm = ChatHuggingFace(llm=llm_endpoint)

In [57]:
prompt = PromptTemplate(
    template="""
You are a helpful assistant.
Answer ONLY from the provided transcript context.
If the context is insufficient, just say you don't know.

{context}
Question: {question}
""",
    input_variables=["context", "question"],
)

In [58]:
question = "Is the topic of aliens discussed in this video? If yes, then what was discussed?"
retrieved_docs = retriever.invoke(question)

# 4 Documents can't go into the prompt as-is; concatenate their page_content
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)

final_prompt = prompt.invoke({"context": context_text, "question": question})
print(final_prompt)

text="\nYou are a helpful assistant.\nAnswer ONLY from the provided transcript context.\nIf the context is insufficient, just say you don't know.\n\nspace age we should have heard a cacophony of voices we should have joined that cacophony of voices and what we did we opened our ears and we heard nothing and many people who argue that there are aliens would say well we haven't really done exhaustive search yet and maybe we're looking in the wrong bands and and we've got the wrong devices and we wouldn't notice what an alien form was like to be so different to what we're used to but you know i'm not i don't really buy that that it shouldn't be as difficult as that like we i think we've searched enough there should be if it were everywhere if it was it should be everywhere we should see dyson's fears being put up sun's blinking in and out you know there should be a lot of evidence for those things and then there are other people argue well the sort of safari view of like well we're a prim

In [59]:
answer = llm.invoke(final_prompt)
print(answer.content)

Yes, the topic of aliens is discussed in this video. 

The discussion revolves around the possibility of alien civilizations, the lack of evidence, and various arguments for and against the existence of extraterrestrial life. Some points discussed include:

1. The speaker's skepticism about the idea that alien civilizations might communicate with humans through mental thoughts or a "universal rule" not to interfere.
2. The idea that if alien civilizations exist, they should be more diverse and complex than just one way of communicating.
3. The possibility of a "violent dictatorship" among successful alien civilizations, leading to a uniformity in their behavior.
4. The concept of the "Great Filter" that might prevent civilizations from advancing further.
5. The speaker's personal opinion, based on their understanding of physics and discussion with experts, that the probability of alien civilizations existing is low, and the most likely scenario is that we are alone.
